In [1]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets openpyxl pandas numpy
print("✅ All dependencies installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 108.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.3 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
from google.colab import files
uploaded=  files.upload()

Mounted at /content/drive


Saving Lending_Loan_Portfolio_1000_Raw.xlsx to Lending_Loan_Portfolio_1000_Raw.xlsx


In [1]:
# ==========================================
# CELL 2: DATA LOADING & FEATURE ENGINEERING
# ==========================================
import pandas as pd
import numpy as np

# 1. Load the raw lending portfolio dataset verbatim
file_name = "Lending_Loan_Portfolio_1000_Raw.xlsx"
try:
    df = pd.read_excel(file_name)
    print(f"Successfully loaded {file_name}. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: Please upload '{file_name}' to your Colab environment root directory.")

# 2. Implement Domain-Specific Feature Engineering
# Feature A: FOIR (Fixed Obligation to Income Ratio) capped at 1.0
df['FOIR'] = df['EMI_AMOUNT'] / df['MONTHLY_INCOME']
df['FOIR'] = df['FOIR'].clip(upper=1.0)

# Feature B: EMI-to-Income Ratio (%)
df['EMI_INCOME_RATIO'] = (df['EMI_AMOUNT'] / df['MONTHLY_INCOME']) * 100

# Feature C: Credit Utilization Ratio (Outstanding / Sanctioned Amount)
# Handling edge case where Sanction Amount might be 0 to avoid division by zero
df['CREDIT_UTILIZATION'] = np.where(df['SANCTION_AMOUNT'] > 0, df['OUTSTANDING_BALANCE'] / df['SANCTION_AMOUNT'], 0)

# Feature D: Delinquency Flag (IS_DELINQUENT: 1 if Current DPD > 0, else 0)
df['IS_DELINQUENT'] = np.where(df['CURRENT_DPD'] > 0, 1, 0)

print("Feature engineering completed. Sample engineered features:")
print(df[['FOIR', 'EMI_INCOME_RATIO', 'CREDIT_UTILIZATION', 'IS_DELINQUENT']].head(3))

Successfully loaded Lending_Loan_Portfolio_1000_Raw.xlsx. Shape: (1000, 24)
Feature engineering completed. Sample engineered features:
       FOIR  EMI_INCOME_RATIO  CREDIT_UTILIZATION  IS_DELINQUENT
0  0.205435         20.543472            0.273016              0
1  0.055799          5.579941            0.773767              0
2  0.481618         48.161800            0.414029              1


In [2]:
# ==========================================
# CELL 3: INSTRUCTION DATASET DESIGN (JSONL)
# ==========================================
import json
from datasets import Dataset

instruction_data = []

for _, row in df.iterrows():
    # Safely handle potential missing values (NaNs) by converting them to strings
    occupation = str(row['OCCUPATION']).lower() if pd.notna(row['OCCUPATION']) else "unknown"

    # Context gathering
    profile_text = (
        f"Borrower Profile:\n"
        f"- Age: {row['AGE']} | Gender: {row['GENDER']} | Occupation: {row['OCCUPATION']}\n"
        f"- Monthly Income: ₹{row['MONTHLY_INCOME']:.2f}\n"
        f"- Bureau Score: {row['BUREAU_SCORE']}\n"
        f"Loan Details:\n"
        f"- Product: {row['LOAN_PRODUCT']} | Requested Amount: ₹{row['LOAN_AMOUNT']:.2f}\n"
        f"- Sanctioned Amount: ₹{row['SANCTION_AMOUNT']:.2f} | Outstanding Balance: ₹{row['OUTSTANDING_BALANCE']:.2f}\n"
        f"- Tenure: {row['LOAN_TENURE']} months | Interest Rate: {row['INTEREST_RATE']}%\n"
        f"- EMI: ₹{row['EMI_AMOUNT']:.2f}\n"
        f"Risk & Repayment Metrics:\n"
        f"- FOIR: {row['FOIR']:.2f} | EMI-to-Income Ratio: {row['EMI_INCOME_RATIO']:.1f}%\n"
        f"- Credit Utilization: {row['CREDIT_UTILIZATION']:.2f}\n"
        f"- Current DPD: {row['CURRENT_DPD']} | Max DPD: {row['MAX_DPD']} | Collection Bucket: {row['COLLECTION_BUCKET']}\n"
    )

    # Task 1: Loan Summary Generation
    summary_completion = (
        f"The borrower is a {row['AGE']}-year-old {occupation} professional. "
        f"With a Bureau Score of {row['BUREAU_SCORE']} and an EMI-to-Income ratio of {row['EMI_INCOME_RATIO']:.1f}%, "
        f"the loan is currently marked as {row['LOAN_STATUS']}. Current DPD stands at {row['CURRENT_DPD']} with a "
        f"Credit Utilization profile of {row['CREDIT_UTILIZATION']:.2f}."
    )
    instruction_data.append({
        "instruction": "Generate a concise domain-aware natural language loan summary based on the borrower profile and repayment behaviors.",
        "input": profile_text,
        "output": summary_completion
    })

    # Task 2: Credit Risk Classification
    risk_label = "Low Risk" if row['BUREAU_SCORE'] >= 750 and row['MAX_DPD'] == 0 else ("High Risk" if row['BUREAU_SCORE'] < 650 or row['MAX_DPD'] > 30 else "Medium Risk")
    instruction_data.append({
        "instruction": "Classify the borrower's credit risk profile into one of these categories: Low Risk, Medium Risk, High Risk.",
        "input": profile_text,
        "output": risk_label
    })

    # Task 3: Loan Approval Recommendation
    rec_label = "Approve" if row['BUREAU_SCORE'] >= 700 and row['FOIR'] <= 0.5 else ("Reject" if row['BUREAU_SCORE'] < 600 or row['FOIR'] > 0.65 else "Approve with Conditions")
    instruction_data.append({
        "instruction": "Provide a definitive loan underwriting decision recommendation: Approve, Approve with Conditions, or Reject.",
        "input": profile_text,
        "output": rec_label
    })

# Save to required format (JSONL)
output_jsonl = "lending_instruction_dataset.jsonl"
with open(output_jsonl, "w") as f:
    for item in instruction_data:
        f.write(json.dumps(item) + "\n")

# Convert to Hugging Face Dataset format
hf_dataset = Dataset.from_list(instruction_data)
print(f"Generated {len(hf_dataset)} prompt-completion pairs and saved to {output_jsonl}")

Generated 3000 prompt-completion pairs and saved to lending_instruction_dataset.jsonl


In [3]:
# ==========================================
# CELL 4: MODEL SELECTION & QUANTIZATION
# ==========================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# Configure 4-bit quantization for consumer hardware constraints
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print(f" Loaded base model: {model_id} under 4-bit precision constraints.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

 Loaded base model: Qwen/Qwen2.5-1.5B-Instruct under 4-bit precision constraints.


In [11]:
# ==========================================
# CELL 5: LORA CONFIGURATION & TRL TRAINING
# ==========================================
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# Define LoRA Configuration targeting core linear projections
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 🛑 THE FIX: Map the dataset manually instead of using formatting_func
def create_text_column(example):
    # This guarantees every row is explicitly converted into a single, clean string
    formatted_text = (
        f"<|im_start|>system\nYou are a domain-aware Lending Intelligence Assistant expert at risk assessment.<|im_end|>\n"
        f"<|im_start|>user\n{example['instruction']}\n\n{example['input']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['output']}<|im_end|>"
    )
    return {"text": formatted_text}

# Apply the formatting to the entire dataset at once
processed_dataset = hf_dataset.map(create_text_column)

# Use SFTConfig for newer TRL versions
training_args = SFTConfig(
    output_dir="./lending_slm_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=120, # Scaled for rapid demonstration & training turnaround
    optim="paged_adamw_8bit",
    fp16=False,
    bf16=True, # Recommended for modern architectures on T4/A100
    warmup_steps=5,
    lr_scheduler_type="cosine",
    save_strategy="no",
    report_to="none",
    max_length=1024,
    dataset_text_field="text" # Tell TRL exactly which column to natively use
)

# Initialize Supervised Fine-Tuning (SFT) Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=processed_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args # Notice we completely removed formatting_func here
)

print("🚀 Starting fine-tuning process...")
trainer.train()
print("✅ Fine-tuning completed successfully!")

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


🚀 Starting fine-tuning process...


Step,Training Loss
10,1.947802
20,0.552652
30,0.399293
40,0.392284
50,0.375463
60,0.374833
70,0.374836
80,0.371769
90,0.364030
100,0.364441


✅ Fine-tuning completed successfully!


In [13]:
# ==========================================
# CELL 6: EVALUATION & INFERENCE DEMO
# ==========================================
import torch

# 1. Switch the model from training mode to evaluation mode
model.eval()

# Test Scenario addressing baseline pain points
test_profile = (
    "Borrower Profile:\n"
    "- Age: 29 | Gender: Female | Occupation: Business\n"
    "- Monthly Income: ₹90,000.00\n"
    "- Bureau Score: 590\n"
    "Loan Details:\n"
    "- Product: Personal Loan | Requested Amount: ₹4,000,000.00\n"
    "- Sanctioned Amount: ₹3,500,000.00 | Outstanding Balance: ₹3,200,000.00\n"
    "- Tenure: 60 months | Interest Rate: 16.5%\n"
    "- EMI: ₹62,000.00\n"
    "Risk & Repayment Metrics:\n"
    "- FOIR: 0.69 | EMI-to-Income Ratio: 68.8%\n"
    "- Credit Utilization: 0.91\n"
    "- Current DPD: 45 | Max DPD: 65 | Collection Bucket: 31-60 days\n"
)

test_prompt = f"<|im_start|>system\nYou are a domain-aware Lending Intelligence Assistant expert at risk assessment.<|im_end|>\n<|im_start|>user\nProvide a definitive loan underwriting decision recommendation: Approve, Approve with Conditions, or Reject.\n\n{test_profile}<|im_end|>\n<|im_start|>assistant\n"

# Run inference
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

# Find the specific token ID for <|im_end|> so the model knows exactly when to stop
eos_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
if eos_token_id is None:
    eos_token_id = tokenizer.eos_token_id

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=15,
        do_sample=False, # Removed temperature to clear the generation flag warning
        eos_token_id=eos_token_id, # Tells the model to stop exactly after its answer
        pad_token_id=tokenizer.pad_token_id
    )

# Extract ONLY the newly generated response (ignoring the input prompt)
input_length = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_length:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("=== INFERENCE RESULT ON TEST PROFILE ===")
print(response)

=== INFERENCE RESULT ON TEST PROFILE ===
Reject


In [14]:
# List of 10 diverse test profiles for evaluation
test_profiles = [
    # Profile 1: IDEAL BORROWER (Expected: Approve)
    # High Bureau, Low FOIR, 0 DPD.
    (
        "Borrower Profile:\n"
        "- Age: 35 | Gender: Male | Occupation: Salaried\n"
        "- Monthly Income: ₹120,000.00\n"
        "- Bureau Score: 810\n"
        "Loan Details:\n"
        "- Product: Home Loan | Requested Amount: ₹5,000,000.00\n"
        "- Sanctioned Amount: ₹5,000,000.00 | Outstanding Balance: ₹4,500,000.00\n"
        "- Tenure: 180 months | Interest Rate: 8.5%\n"
        "- EMI: ₹49,000.00\n"
        "Risk & Repayment Metrics:\n"
        "- FOIR: 0.41 | EMI-to-Income Ratio: 40.8%\n"
        "- Credit Utilization: 0.90\n"
        "- Current DPD: 0 | Max DPD: 0 | Collection Bucket: Current\n"
    ),

    # Profile 2: BORDERLINE RISK (Expected: Approve with Conditions)
    # Moderate Bureau (680), occasional slight delays (DPD 15).
    (
        "Borrower Profile:\n"
        "- Age: 42 | Gender: Female | Occupation: Self-Employed\n"
        "- Monthly Income: ₹85,000.00\n"
        "- Bureau Score: 680\n"
        "Loan Details:\n"
        "- Product: MSME Loan | Requested Amount: ₹1,500,000.00\n"
        "- Sanctioned Amount: ₹1,200,000.00 | Outstanding Balance: ₹800,000.00\n"
        "- Tenure: 48 months | Interest Rate: 12.0%\n"
        "- EMI: ₹31,000.00\n"
        "Risk & Repayment Metrics:\n"
        "- FOIR: 0.36 | EMI-to-Income Ratio: 36.5%\n"
        "- Credit Utilization: 0.67\n"
        "- Current DPD: 15 | Max DPD: 30 | Collection Bucket: 1-30 days\n"
    ),

    # Profile 3: FINANCIAL STRESS (Expected: Reject)
    # Good Bureau, but FOIR is extremely high (> 0.65), indicating unaffordability.
    (
        "Borrower Profile:\n"
        "- Age: 28 | Gender: Male | Occupation: Salaried\n"
        "- Monthly Income: ₹45,000.00\n"
        "- Bureau Score: 710\n"
        "Loan Details:\n"
        "- Product: Personal Loan | Requested Amount: ₹800,000.00\n"
        "- Sanctioned Amount: ₹800,000.00 | Outstanding Balance: ₹750,000.00\n"
        "- Tenure: 60 months | Interest Rate: 14.0%\n"
        "- EMI: ₹31,000.00\n"
        "Risk & Repayment Metrics:\n"
        "- FOIR: 0.69 | EMI-to-Income Ratio: 68.9%\n"
        "- Credit Utilization: 0.94\n"
        "- Current DPD: 0 | Max DPD: 15 | Collection Bucket: Current\n"
    ),

    # Profile 4: DELINQUENT HISTORY (Expected: Reject)
    # Very poor Bureau (580), high delinquency (75 DPD).
    (
        "Borrower Profile:\n"
        "- Age: 50 | Gender: Male | Occupation: Business\n"
        "- Monthly Income: ₹150,000.00\n"
        "- Bureau Score: 580\n"
        "Loan Details:\n"
        "- Product: Vehicle Loan | Requested Amount: ₹2,000,000.00\n"
        "- Sanctioned Amount: ₹1,800,000.00 | Outstanding Balance: ₹1,600,000.00\n"
        "- Tenure: 60 months | Interest Rate: 11.0%\n"
        "- EMI: ₹39,000.00\n"
        "Risk & Repayment Metrics:\n"
        "- FOIR: 0.26 | EMI-to-Income Ratio: 26.0%\n"
        "- Credit Utilization: 0.89\n"
        "- Current DPD: 75 | Max DPD: 90 | Collection Bucket: 61-90 days\n"
    ),

    # Profile 5: PERFECT FILE (Expected: Approve)
    # Exceptional Bureau (850), highly affordable (FOIR 0.15).
    (
        "Borrower Profile:\n"
        "- Age: 30 | Gender: Female | Occupation: Salaried\n"
        "- Monthly Income: ₹200,000.00\n"
        "- Bureau Score: 850\n"
        "Loan Details:\n"
        "- Product: Vehicle Loan | Requested Amount: ₹1,200,000.00\n"
        "- Sanctioned Amount: ₹1,200,000.00 | Outstanding Balance: ₹600,000.00\n"
        "- Tenure: 48 months | Interest Rate: 9.0%\n"
        "- EMI: ₹29,800.00\n"
        "Risk & Repayment Metrics:\n"
        "- FOIR: 0.15 | EMI-to-Income Ratio: 14.9%\n"
        "- Credit Utilization: 0.50\n"
        "- Current DPD: 0 | Max DPD: 0 | Collection Bucket: Current\n"
    ),

    # Profile 6: HISTORICAL DELAYS (Expected: Approve with Conditions)
    # Good Bureau, 0 current DPD, but has a history of max DPD 45.
    (
        "Borrower Profile:\n"
        "- Age: 38 | Gender: Male | Occupation: Business\n"
        "- Monthly Income: ₹70,000.00\n"
        "- Bureau Score: 710\n"
        "Loan Details:\n"
        "- Product: Personal Loan | Requested Amount: ₹600,000.00\n"
        "- Sanctioned Amount: ₹500,000.00 | Outstanding Balance: ₹480,000.00\n"
        "- Tenure: 36 months | Interest Rate: 15.0%\n"
        "- EMI: ₹17,300.00\n"
        "Risk & Repayment Metrics:\n"
        "- FOIR: 0.25 | EMI-to-Income Ratio: 24.7%\n"
        "- Credit Utilization: 0.96\n"
        "- Current DPD: 0 | Max DPD: 45 | Collection Bucket: Current\n"
    ),

    # Profile 7: POTENTIAL DEFAULT (Expected: Reject)
    # High risk Bureau (540), severe delinquency (>90 DPD).
    (
        "Borrower Profile:\n"
        "- Age: 45 | Gender: Female | Occupation: Self-Employed\n"
        "- Monthly Income: ₹60,000.00\n"
        "- Bureau Score: 540\n"
        "Loan Details:\n"
        "- Product: Consumer Durable Loan | Requested Amount: ₹100,000.00\n"
        "- Sanctioned Amount: ₹100,000.00 | Outstanding Balance: ₹80,000.00\n"
        "- Tenure: 12 months | Interest Rate: 18.0%\n"
        "- EMI: ₹9,100.00\n"
        "Risk & Repayment Metrics:\n"
        "- FOIR: 0.15 | EMI-to-Income Ratio: 15.2%\n"
        "- Credit Utilization: 0.80\n"
        "- Current DPD: 95 | Max DPD: 120 | Collection Bucket: 90+ days\n"
    ),

    # Profile 8: THIN FILE / YOUNG (Expected: Approve with Conditions)
    # Young applicant, moderate Bureau (660), slight current delay (DPD 5).
    (
        "Borrower Profile:\n"
        "- Age: 23 | Gender: Male | Occupation: Salaried\n"
        "- Monthly Income: ₹35,000.00\n"
        "- Bureau Score: 660\n"
        "Loan Details:\n"
        "- Product: Consumer Durable Loan | Requested Amount: ₹50,000.00\n"
        "- Sanctioned Amount: ₹45,000.00 | Outstanding Balance: ₹20,000.00\n"
        "- Tenure: 6 months | Interest Rate: 0.0%\n"
        "- EMI: ₹7,500.00\n"
        "Risk & Repayment Metrics:\n"
        "- FOIR: 0.21 | EMI-to-Income Ratio: 21.4%\n"
        "- Credit Utilization: 0.44\n"
        "- Current DPD: 5 | Max DPD: 15 | Collection Bucket: 1-30 days\n"
    ),

    # Profile 9: BORDERLINE AFFORDABILITY (Expected: Approve with Conditions)
    # High Bureau (760), but FOIR is just over the strict 0.5 threshold for pure approval.
    (
        "Borrower Profile:\n"
        "- Age: 33 | Gender: Female | Occupation: Salaried\n"
        "- Monthly Income: ₹110,000.00\n"
        "- Bureau Score: 760\n"
        "Loan Details:\n"
        "- Product: Home Loan | Requested Amount: ₹7,000,000.00\n"
        "- Sanctioned Amount: ₹6,500,000.00 | Outstanding Balance: ₹6,400,000.00\n"
        "- Tenure: 240 months | Interest Rate: 8.75%\n"
        "- EMI: ₹57,000.00\n"
        "Risk & Repayment Metrics:\n"
        "- FOIR: 0.52 | EMI-to-Income Ratio: 51.8%\n"
        "- Credit Utilization: 0.98\n"
        "- Current DPD: 0 | Max DPD: 0 | Collection Bucket: Current\n"
    ),

    # Profile 10: HIGH INCOME / HIGH DELINQUENCY (Expected: Reject)
    # Makes good money, but has a poor repayment track record (Bureau 610, DPD 65).
    (
        "Borrower Profile:\n"
        "- Age: 55 | Gender: Male | Occupation: Business\n"
        "- Monthly Income: ₹350,000.00\n"
        "- Bureau Score: 610\n"
        "Loan Details:\n"
        "- Product: MSME Loan | Requested Amount: ₹10,000,000.00\n"
        "- Sanctioned Amount: ₹8,500,000.00 | Outstanding Balance: ₹7,000,000.00\n"
        "- Tenure: 120 months | Interest Rate: 13.5%\n"
        "- EMI: ₹129,000.00\n"
        "Risk & Repayment Metrics:\n"
        "- FOIR: 0.37 | EMI-to-Income Ratio: 36.9%\n"
        "- Credit Utilization: 0.82\n"
        "- Current DPD: 65 | Max DPD: 85 | Collection Bucket: 61-90 days\n"
    )
]

In [15]:
for idx, profile in enumerate(test_profiles, 1):
    prompt = f"<|im_start|>system\nYou are a domain-aware Lending Intelligence Assistant expert at risk assessment.<|im_end|>\n<|im_start|>user\nProvide a definitive loan underwriting decision recommendation: Approve, Approve with Conditions, or Reject.\n\n{profile}<|im_end|>\n<|im_start|>assistant\n"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=15,
            do_sample=False,
            eos_token_id=eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    print(f"Profile {idx} Recommendation: {response}")

Profile 1 Recommendation: Approve with Conditions
Profile 2 Recommendation: Approve with Conditions
Profile 3 Recommendation: Approve with Conditions
Profile 4 Recommendation: Reject
Profile 5 Recommendation: Approve with Conditions
Profile 6 Recommendation: Approve with Conditions
Profile 7 Recommendation: Approve with Conditions
Profile 8 Recommendation: Approve with Conditions
Profile 9 Recommendation: Approve with Conditions
Profile 10 Recommendation: Approve with Conditions
